# HMM route decoding: design and evaluation
The hidden state at each GPS point is a directed candidate arc. The decoder sees GPS, candidate probabilities and the road network; it never receives a reference route. Reference routes are read only by the offline evaluation code.


The classifier already uses distance, direction, speed and other features. Its scores are normalised across the candidate arcs at a point. We multiply the log of that normalised probability by alpha, using a small floor to avoid the log of zero. We retain the top-k candidates for speed and report how often pruning removes an available label.


For a pair of candidates, we compare directed travel distance on the network against straight-line GPS displacement. A larger difference costs more. Reverse-direction twins receive an additional penalty; small backwards projection movement on the same arc is allowed as GPS jitter. Search is bounded using the actual elapsed time. Long gaps get a wider search and a softer distance penalty. If no pair is reachable, we restart the chain and count a break; if only some are unreachable they receive a very low score. Missing points carry the previous arc, except that an empty first point must raise an error because there is no previous arc.


## Viterbi recurrence
For candidate j at time t, `score[t,j] = emission[t,j] + max_i(score[t-1,i] + transition[t,i,j])`. Store the maximizing i as a back-pointer. Start each segment with its emission scores, then backtrack from the highest final score. This uses log probabilities, avoiding repeated multiplication of small numbers.


## Experimental design
Training parameters use five trip-grouped out-of-fold predictions from the untuned model. The final seed-42 sample covers alpha {0.5, 1, 2}, beta {15, 30, 60} m, U-turn cost {0, 2, 5}, gap multiplier {2, 3}, minimum cutoff {50, 100} m, and top-k {10, 15}. These coarse values span weaker and stronger reliance on classifier and network evidence while keeping the search inexpensive; the gap multiplier remains above one to preserve relaxation. A seeded sample of combinations is evaluated on training trips spanning sizes, with gap-heavy and far-heavy examples. Student-test is held out until parameters are frozen. The training smoke run is a functional check, not an unbiased score. Earlier exploratory runs are preserved and excluded from final selection because one allowed no gap relaxation and another passed unused label columns to the decoder.


In [ ]:
from pathlib import Path
import pandas as pd
here = Path.cwd()
pd.read_csv(here / 'results' / 'decoder_trials.csv').head()


In [ ]:
pd.read_csv(here / 'results' / 'per_trip_metrics.csv').head()


See `results/REPORT_ANSWERS.md` for the completed results, uncertainty, limitations and trips where the HMM did not help. The saved tables, figures and maps contain the measurements from the actual run.
